# Topic Modeling
## This notebook outlines the concepts involved in Topic Modeling


Topic modeling is a statistical model to **discover** the abstract "topics" that occur in a collection of documents

It is commonly used in text document. But nowadays, in social media analysis, topic modeling is an emerging research area.

One of the most popular algorithms used is **Latent Dirichlet Allocation** which was proposed by
David Blei et al in 2003.

Dataset:
https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Tokenize
    - Stop words removal
    - Non-alphabetic words removal
    - Lowercase
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Visualize the topics

### Install the necessary library

In [1]:
! pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.1 MB/s eta 0:00:00


In [2]:
import nltk
! nltk.download('stopwords')

/bin/bash: -c: line 1: syntax error near unexpected token `'stopwords''
/bin/bash: -c: line 1: ` nltk.download('stopwords')'


### Import the necessary libraries

In [3]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import LdaModel
from gensim.corpora import Dictionary
from pprint import pprint
import pandas as pd
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer
from gensim import corpora, models
import gensim

### Download the dataset

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/subashgandyer/datasets/main/kaggledatasets.csv"
df = pd.read_csv(url)
df.head()



,Title,Subtitle,Owner,Votes,Versions,Tags,Data Type,Size,License,Views,Download,Kernels,Topics,URL,Description
0,Credit Card Fraud Detection,Anonymized credit card transactions labeled as...,Machine Learning Group - ULB,1241,"Version 2,2016-11-05|Version 1,2016-11-03",crime\nfinance,CSV,144 MB,ODbL,"442,136 views","53,128 downloads","1,782 kernels",26 topics,https://www.kaggle.com/mlg-ulb/creditcardfraud,The datasets contains transactions made by cre...
1,European Soccer Database,"25k+ matches, players & teams attributes for E...",Hugo Mathien,1046,"Version 10,2016-10-24|Version 9,2016-10-24|Ver...",association football\neurope,SQLite,299 MB,ODbL,"396,214 views","46,367 downloads","1,459 kernels",75 topics,https://www.kaggle.com/hugomathien/soccer,The ultimate Soccer database for data analysis...
2,TMDB 5000 Movie Dataset,"Metadata on ~5,000 movies from TMDb",The Movie Database (TMDb),1024,"Version 2,2017-09-28",film,CSV,44 MB,Other,"446,255 views","62,002 downloads","1,394 kernels",46 topics,https://www.kaggle.com/tmdb/tmdb-movie-metadata,Background\nWhat can we say about the success ...
3,Global Terrorism Database,"More than 170,000 terrorist attacks worldwide,...",START Consortium,789,"Version 2,2017-07-19|Version 1,2016-12-08",crime\nterrorism\ninternational relations,CSV,144 MB,Other,"187,877 views","26,309 downloads",608 kernels,11 topics,https://www.kaggle.com/START-UMD/gtd,"Context\nInformation on more than 170,000 Terr..."
4,Bitcoin Historical Data,Bitcoin data at 1-min intervals from select ex...,Zielak,618,"Version 11,2018-01-11|Version 10,2017-11-17|Ve...",history\nfinance,CSV,119 MB,CC4,"146,734 views","16,868 downloads",68 kernels,13 topics,https://www.kaggle.com/mczielinski/bitcoin-his...,Context\nBitcoin is the longest running and mo...


### Load the dataset

In [5]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (2150, 15)
Columns: ['Title', 'Subtitle', 'Owner', 'Votes', 'Versions', 'Tags', 'Data Type', 'Size', 'License', 'Views', 'Download', 'Kernels', 'Topics', 'URL', 'Description']


### Explore the dataset

In [6]:
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
df.describe()

Title          object
Subtitle       object
Owner          object
Votes           int64
Versions       object
Tags           object
Data Type      object
Size           object
License        object
Views          object
Download       object
Kernels        object
Topics         object
URL            object
Description    object
dtype: object

Missing values:
Title            0
Subtitle       104
Owner            0
Votes            0
Versions         5
Tags           542
Data Type        0
Size             0
License          0
Views            5
Download        15
Kernels        944
Topics         592
URL              0
Description      5
dtype: int64


,Votes
count,2150.000000
mean,24.011628
std,64.788465
min,2.000000
25%,4.000000
50%,8.000000
75%,19.000000
max,1241.000000


### Extract the data for topic modeling

In [7]:
# Extract the Description column for topic modeling
# Drop rows with missing descriptions
docs = df['Description'].dropna().tolist()

### Pre-process the dataset
- Tokenize
- Stop words removal
- Non-alphabetic words removal
- Lowercase
- Define them

### Define the pattern, tokenizer, stop words and lemmatizer

In [9]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [10]:
pattern = r'\w+'
tokenizer = RegexpTokenizer(pattern)
en_stop = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

### Preprocess

In [12]:
texts = []


for idx, doc in df['Description'].items():
    if pd.isna(doc):
        continue
    # clean and tokenize document string
    raw = doc.lower()
    tokens = tokenizer.tokenize(raw)

    # remove stop words from tokens
    tokens = [t for t in tokens if not t in en_stop]

    # lemmatize tokens

    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    # remove word containing only single char
    tokens = [t for t in tokens if len(t)>1]

    # add tokens to list
    texts.append(tokens)


print(texts[0])

['datasets', 'contains', 'transaction', 'made', 'credit', 'card', 'september', '2013', 'european', 'cardholder', 'dataset', 'present', 'transaction', 'occurred', 'two', 'day', '492', 'fraud', '284', '807', 'transaction', 'dataset', 'highly', 'unbalanced', 'positive', 'class', 'fraud', 'account', '172', 'transaction', 'contains', 'numerical', 'input', 'variable', 'result', 'pca', 'transformation', 'unfortunately', 'due', 'confidentiality', 'issue', 'cannot', 'provide', 'original', 'feature', 'background', 'information', 'data', 'feature', 'v1', 'v2', 'v28', 'principal', 'component', 'obtained', 'pca', 'feature', 'transformed', 'pca', 'time', 'amount', 'feature', 'time', 'contains', 'second', 'elapsed', 'transaction', 'first', 'transaction', 'dataset', 'feature', 'amount', 'transaction', 'amount', 'feature', 'used', 'example', 'dependant', 'cost', 'senstive', 'learning', 'feature', 'class', 'response', 'variable', 'take', 'value', 'case', 'fraud', 'otherwise', 'given', 'class', 'imbalanc

### Create a dictionary

In [13]:
dictionary = Dictionary(texts)

### Filter low frequency words

In [14]:
dictionary.filter_extremes(no_below=2, no_above=0.5)

### Convert tokenized documents into a document-term matrix

In [15]:
corpus = [dictionary.doc2bow(text) for text in texts]

### Create an index to word dictionary

In [16]:
id2word = {v: k for k, v in dictionary.token2id.items()}

### Train the Topic model

In [17]:
# Train LDA Topic Model
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=15,
    random_state=42,
    update_every=1,
    chunksize=100,
    passes=10,
    alpha='auto',
    per_word_topics=True

)

### Display the topics

In [18]:
topics = lda_model.print_topics(num_words=5)
for topic in topics:
    print(topic)

(0, '0.037*"file" + 0.022*"csv" + 0.018*"http" + 0.017*"com" + 0.015*"inspiration"')
(1, '0.070*"team" + 0.061*"player" + 0.057*"game" + 0.040*"match" + 0.031*"score"')
(2, '0.117*"description" + 0.113*"yet" + 0.083*"review" + 0.049*"deep" + 0.039*"train"')
(3, '0.043*"submission" + 0.039*"attack" + 0.038*"defense" + 0.037*"cell" + 0.029*"patient"')
(4, '0.060*"college" + 0.051*"school" + 0.039*"student" + 0.036*"numeric" + 0.032*"layer"')
(5, '0.031*"time" + 0.013*"day" + 0.011*"column" + 0.011*"one" + 0.010*"date"')
(6, '0.030*"state" + 0.019*"information" + 0.013*"open" + 0.012*"database" + 0.012*"2016"')
(7, '0.047*"country" + 0.027*"bureau" + 0.027*"rate" + 0.027*"food" + 0.021*"statistic"')
(8, '0.037*"survey" + 0.034*"health" + 0.031*"variable" + 0.029*"center" + 0.019*"social"')
(9, '0.111*"price" + 0.048*"market" + 0.045*"stock" + 0.029*"transaction" + 0.020*"dog"')
(10, '0.047*"year" + 0.029*"total" + 0.028*"per" + 0.027*"number" + 0.016*"population"')
(11, '0.023*"age" + 0.0

### Display the 15 topics with words

In [22]:
print("=== 15 Topics with Top words ===\n")
for idx, topic in lda_model.print_topics(num_topics=15, num_words=10):
  print(f"Topic #{idx}:")
  print(f"  {topic}")
  print()

=== 15 Topics with Top words ===

Topic #0:
  0.037*"file" + 0.022*"csv" + 0.018*"http" + 0.017*"com" + 0.015*"inspiration" + 0.015*"contains" + 0.013*"name" + 0.012*"user" + 0.011*"id" + 0.011*"question"

Topic #1:
  0.070*"team" + 0.061*"player" + 0.057*"game" + 0.040*"match" + 0.031*"score" + 0.029*"season" + 0.028*"street" + 0.020*"sport" + 0.019*"point" + 0.018*"vehicle"

Topic #2:
  0.117*"description" + 0.113*"yet" + 0.083*"review" + 0.049*"deep" + 0.039*"train" + 0.031*"positive" + 0.025*"audio" + 0.024*"music" + 0.022*"negative" + 0.021*"else"

Topic #3:
  0.043*"submission" + 0.039*"attack" + 0.038*"defense" + 0.037*"cell" + 0.029*"patient" + 0.029*"targeted" + 0.022*"visual" + 0.020*"damage" + 0.018*"pokemon" + 0.018*"cancer"

Topic #4:
  0.060*"college" + 0.051*"school" + 0.039*"student" + 0.036*"numeric" + 0.032*"layer" + 0.027*"education" + 0.018*"grade" + 0.017*"device" + 0.016*"binary" + 0.013*"mat"

Topic #5:
  0.031*"time" + 0.013*"day" + 0.011*"column" + 0.011*"one" 

### LSI Model
- Import model
- Train the model
- Display the topics

In [23]:
from gensim.models import LsiModel

#Train LSI model
lsi_model=LsiModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=15
)
print("LSI model trained successfully!")

LSI model trained successfully!


In [26]:
!pip install pyLDAvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 30.8 MB/s eta 0:00:00


## Visualize the topics and documents with the trained Topic Model
- Use pyLDAvis from gensim

In [27]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Enable the notebook for visualization

In [28]:
pyLDAvis.enable_notebook()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Visualize the Topic model

In [29]:
# Prepare the interactive visualization
vis_data = gensimvis.prepare(lda_model, corpus, dictionary)
vis_data

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
5     -0.237661 -0.104131       1        1  20.032289
0     -0.285037  0.033289       2        1  16.891481
6     -0.214103 -0.019748       3        1  16.421333
12    -0.218828  0.089474       4        1  14.565903
10    -0.048964 -0.217655       5        1   6.895763
11     0.023764 -0.069938       6        1   4.663087
13    -0.055661  0.331743       7        1   4.619166
14     0.045197 -0.136754       8        1   3.157676
8      0.093442 -0.009697       9        1   2.944705
1      0.100893  0.017191      10        1   2.601865
7      0.135900 -0.017364      11        1   2.043317
4      0.164236  0.032077      12        1   1.763568
9      0.162227  0.019348      13        1   1.366812
2      0.173870  0.036727      14        1   1.061418
3      0.160725  0.015439      15        1   0.971617, topic_info=             Term         Freq        Total Category  logprob  loglift
727    university  2118.000000  2118.000000  Default  30.0000  30.0000
995   description   738.000000   738.000000  Default  29.0000  29.0000
363          file  1923.000000  1923.000000  Default  28.0000  28.0000
1037         year  1368.000000  1368.000000  Default  27.0000  27.0000
1168         word   803.000000   803.000000  Default  26.0000  26.0000
...           ...          ...          ...      ...      ...      ...
646           non    38.211390   413.014138  Topic15  -4.2698   2.2536
1399        image    47.129976   950.302931  Topic15  -4.0600   1.6301
129          base    19.700814   104.469411  Topic15  -4.9322   2.9657
385            id    20.459479   634.294351  Topic15  -4.8944   1.1999
2286   classified    17.244750    45.044980  Topic15  -5.0654   3.6738

[776 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
739       5  0.991316       01
106       1  0.048885       10
106       2  0.003492       10
106       3  0.092532       10
106       4  0.530751       10
...     ...       ...      ...
2468      1  0.991539      yes
490       5  0.081208      yet
490      14  0.914939      yet
2819      8  0.992615     york
2684      8  0.978947  youtube

[1273 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[6, 1, 7, 13, 11, 12, 14, 15, 9, 2, 8, 5, 10, 3, 4])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
# Save visualization to HTML file
pyLDAvis.save_html(vis_data, 'topic_model_visualization.html')
print("Visualization saved successfully")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Visualization saved successfully


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag